## Phase 2：原始 Qwen 模型基线与物流任务评测

这一阶段要回答一个问题：**未经 FleetMind 数据微调的 Qwen3-0.6B，完成 Task A 和 Task B 的能力如何？** 我们会固定一套内部评测规则，记录它的成绩。后续 SFT + LoRA 模型按相同规则评测，才能判断提升是否真实。

Phase 1 已完成 2,000 条指令样本：训练 1,600 条、验证 200 条、测试 200 条。**Phase 2 先用验证集**；测试集留到模型方案确定后做最终比较。

| 步骤 | 我们要做什么 | 用到的技术知识及作用 |
|---|---|---|
| **1. 固定评测规则** | 明确 Task A、B 各评什么，确定模型版本、提示词和生成参数 | **Benchmark 设计、数据隔离**：让微调前后面对同样的题目和评分规则，避免比较不公平 |
| **2. 加载原始模型** | 在 Colab T4 上加载 Qwen3-0.6B 和 tokenizer | **Tokenizer、模型推理、GPU 显存**：理解文字如何变成 token，以及模型如何逐步生成答案 |
| **3. 检查少量输出** | 各取几条 A、B 样本，只把题目给模型，查看回答 | **Chat Messages、提示词设计**：确认输入格式正确，尤其不能把标准答案一起传给模型 |
| **4. 运行验证集基线** | 对 200 条验证样本生成回答，保存每条预测 | **批量推理、可复现性**：固定生成设置，留下可复查的原始结果 |
| **5. 评分并分析错误** | 分任务计算指标，查看典型错误 | **结构化输出解析、分类指标、误差分析**：知道模型错在格式、字段提取，还是风险判断 |

### 我们具体看哪些指标？

- **Task A｜物流信息提取：**JSON 有效率、字段准确率；同时检查漏字段、错填字段和凭空编造的信息。
- **Task B｜风险分类：**Accuracy、Precision、Recall、F1；特别关注不同类别的错误，避免只看总体准确率。
- **运行记录：**模型版本、提示词、生成参数、推理耗时和逐条预测。这些是以后复现成绩的依据。

这里的 **benchmark 是我们固定的物流评测样本与评分规则**；**baseline 是原始 Qwen3-0.6B 在这套规则下取得的分数**。Phase 2 只做推理评测，尚未进行 SFT 或 LoRA 训练，因此此阶段没有 Training Loss。训练与验证 Loss 会在 Phase 3 记录。

**阶段交付物**是一份基线指标表、一份逐条预测记录和一份典型错误清单。下一步我们就从“固定评测规则”开始，先把 Task A、B 的评分口径定清楚，再运行模型。

# 运行环境检查

本步骤确认当前 Colab 会话已连接 NVIDIA GPU。Phase 0 曾验证过 T4 可用，但新的 Colab 会话仍需检查一次。后续加载 Qwen3-0.6B 和运行推理会使用这块 GPU。

完成标准：`GPU available` 显示 `True`，并显示 GPU 名称。

In [ ]:
import torch

print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
else:
    print("请在 Runtime → Change runtime type 中选择 T4 GPU，然后重新运行。")

# 1：查看验证集格式

## 这一步做什么？

上传 Phase 1 的正式版数据集，读取 `val.jsonl`，查看样本和标准答案的格式。

## 为什么先做这一步？

接下来要评测 Qwen3-0.6B。我们需要先弄清楚：

- Task A 的标准 JSON 有哪些字段；
- Task B 的风险类别如何表示；
- 每条样本的提问和标准答案分别存在哪里。

弄清楚这些后，才能写出正确的评分代码。本步骤只查看数据，不运行或训练模型。

## 完成标准

成功读到 200 条验证样本，并显示前两条样本的内容。

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile
import json

# 选择 FleetMind_LLM_Phase1_Dataset.zip
uploaded = files.upload()
zip_name = next(name for name in uploaded if name.endswith(".zip"))

data_dir = Path("/content/fleetmind_phase1")
data_dir.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_name) as zf:
    zf.extractall(data_dir)

val_files = list(data_dir.rglob("val.jsonl"))
assert len(val_files) == 1, f"找到 {len(val_files)} 个 val.jsonl"

with val_files[0].open(encoding="utf-8") as f:
    val_data = [json.loads(line) for line in f if line.strip()]

print("验证集样本数：", len(val_data))
assert len(val_data) == 200, "验证集样本数与 Phase 1 正式版不一致"

print("顶层字段：", list(val_data[0].keys()))

for i, sample in enumerate(val_data[:2], start=1):
    print(f"\n===== 样本 {i} =====")
    print(json.dumps(sample, ensure_ascii=False, indent=2)[:5000])

验证集读取成功，200 条样本、格式与 Phase 1 正式版一致。现在我们知道：
- messages 中的 system + user 是给模型的题目；
- assistant 是标准答案，只用于评分，不能放进模型输入；
- Task A 要提取 8 个字段；Task B 要输出 delay_label，取值为 delayed 或 on_time；
- 同一条运单有 A、B 两道题，所以 200 条指令不等于 200 条独立运单。
接下来先做一个很小的数据统计，确认 A/B 数量、风险标签分布和缺失字段样本数量。这样我们就能把评分规则定下来。

# 2：确认评测集组成

## 这一步做什么？

统计验证集中 Task A、Task B 的样本数，查看 Task B 的标签分布，以及 Task A 有多少条包含缺失字段。

## 为什么做？

如果两类任务或风险标签的数量不均衡，单看总体准确率可能产生误导。我们也要单独关注 Task A 遇到缺失信息时，模型能否正确填写 `null`。

## 关键规则

模型输入只包含 `system` 和 `user` 消息；`assistant` 消息是标准答案，仅用于评分。

In [ ]:
from collections import Counter
import json

# 分别统计任务数量、风险标签数量
task_counts = Counter()
label_counts = Counter()

# 统计 Task A 中需要把缺失信息填为 null 的样本
a_with_missing = 0

# 用运单编号统计独立运单数
source_ids = set()

for sample in val_data:
    # 读取当前样本的任务类型：A（信息提取）或 B（延误分类）
    task = sample["task"]
    task_counts[task] += 1
    source_ids.add(sample["source_load_id"])

    # 确认三条消息依次为：系统要求、用户题目、标准答案
    messages = sample["messages"]
    assert [m["role"] for m in messages] == [
        "system", "user", "assistant"
    ]

    # assistant 消息是标准答案，只用于统计和评分
    # 后续调用模型时，绝不能把它放入输入
    gold = json.loads(messages[2]["content"])

    if task == "A":
        # omitted_fields 非空，表示题目故意没有提供某些字段
        if sample["omitted_fields"]:
            a_with_missing += 1

    elif task == "B":
        # 统计 delayed 和 on_time 各有多少条
        label_counts[gold["delay_label"]] += 1

    else:
        raise ValueError(f"未知任务：{task}")

# 展示评测集的组成
print("任务数量：", dict(task_counts))
print("独立运单数：", len(source_ids))
print("Task A 含缺失字段的样本数：", a_with_missing)
print("Task B 标签分布：", dict(label_counts))

# 3. 固定评分规则

## Task A：物流信息提取

要求模型只输出一个 JSON 对象，包含指定的 8 个字段。

记录以下指标：

- **JSON 有效率**：回答能否直接解析为 JSON 对象。
- **结构正确率**：JSON 是否恰好包含要求的 8 个字段。
- **字段准确率**：8 个字段中，有多少个值与标准答案一致。
- **缺失字段准确率**：题目未提供的信息，模型是否正确填写 `null`。

字段准确率以全部 100 条 A 样本的 800 个字段为分母。无效 JSON 的字段记为错误，不从分母中剔除。

## Task B：延误分类

要求模型只输出 `{"delay_label": "delayed"}` 或
`{"delay_label": "on_time"}`。

记录 Accuracy、Macro-F1，以及 delayed 和 on_time 各自的 Precision、Recall、F1。无效或不在允许范围内的输出记为错误。

## 比较原则

原始 Qwen 与之后的 LoRA 模型使用相同的数据、输入格式、生成设置和评分代码。验证集用于开发和比较；测试集留到最终方案确定后使用。

In [ ]:
import json

# Task A 必须提取的 8 个字段；字段名与 Phase 1 标准答案一致
A_FIELDS = (
    "load_id",
    "truck_id",
    "origin_city",
    "destination_city",
    "revenue_usd",
    "fuel_used_gallons",
    "scheduled_delivery",
    "actual_delivery",
)

# Task B 允许输出的两种延误标签
B_LABELS = ("delayed", "on_time")

# 检查验证集的标准答案是否符合刚才确定的评分规则
for sample in val_data:
    # 第三条 assistant 消息存放标准答案
    gold = json.loads(sample["messages"][2]["content"])

    if sample["task"] == "A":
        assert set(gold.keys()) == set(A_FIELDS), (
            f"Task A 字段不一致：{sample['sample_id']}"
        )
    else:
        assert set(gold.keys()) == {"delay_label"}, (
            f"Task B 字段不一致：{sample['sample_id']}"
        )
        assert gold["delay_label"] in B_LABELS, (
            f"Task B 标签无效：{sample['sample_id']}"
        )

print("评分规则与 200 条标准答案一致")
print("Task A 字段数：", len(A_FIELDS))
print("Task B 允许标签：", B_LABELS)
print("只预测 delayed 的参考 Accuracy：67%")

# 4. 加载原始 Qwen3-0.6B

本步骤在 Colab 的 Tesla T4 上加载未经 FleetMind 数据微调的 Qwen3-0.6B。

Tokenizer 负责把文字转换成模型能处理的 token；模型负责根据输入生成回答。我们使用 float16 减少显存占用，并切换到评估模式。

完成标准：输出模型名称、运行设备和参数精度，且运行设备为 CUDA。

In [ ]:
import torch
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM

# 固定本次基线使用的模型名称
MODEL_ID = "Qwen/Qwen3-0.6B"

# 确认当前会话仍连接 GPU
assert torch.cuda.is_available(), "当前没有可用 GPU，请检查 Colab 运行时设置"

# 加载分词器：负责将聊天内容转换为 token
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# 加载未经本项目微调的原始模型
# float16 适合当前 T4，device_map="cuda" 将模型放到 GPU
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,
    device_map="cuda",
)

# 切换到评估模式
model.eval()

# 记录环境与模型信息，便于之后复现实验
print("模型：", MODEL_ID)
print("Transformers：", transformers.__version__)
print("PyTorch：", torch.__version__)
print("运行设备：", model.device)
print("参数精度：", next(model.parameters()).dtype)

#5. 试运行两类物流任务

从验证集各选一条 Task A 和 Task B 样本，让原始 Qwen3-0.6B 生成回答，并与标准答案对照。

模型只能看到 system 和 user 消息。assistant 消息是标准答案，生成时不传给模型。

本次使用确定性生成，并关闭思考模式。完成标准：两类任务都能生成回答；此时先观察格式和内容，不计算整体成绩。

In [ ]:
import torch

# 固定本阶段的生成设置；后续比较 LoRA 模型时也使用同一设置
MAX_NEW_TOKENS = 256

def generate_answer(sample):
    # 只取系统要求和用户题目，绝不传入 assistant 标准答案
    prompt_messages = sample["messages"][:2]

    # 按 Qwen 的聊天模板将消息转换为模型输入
    inputs = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=True,
        add_generation_prompt=True,
        enable_thinking=False,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)

    # 评测时不计算梯度，减少显存占用
    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    # 去掉输入部分的 token，只解码模型新生成的回答
    new_tokens = output_ids[0, inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

# 每类任务各选第一条验证样本
for task in ("A", "B"):
    sample = next(s for s in val_data if s["task"] == task)
    prediction = generate_answer(sample)

    print(f"\n===== Task {task}：{sample['sample_id']} =====")
    print("模型回答：", prediction)
    print("标准答案：", sample["messages"][2]["content"])

- Task A：8 个字段的值都与标准答案一致，但模型加了 ```json 代码围栏。按我们预先定好的“直接输出合法 JSON”规则，JSON 有效率这一项记为失败；后续可以另记“去掉围栏后字段是否正确”，用于分析错误原因，不能悄悄放宽主指标。
- Task B：预测错误。实际交付是 21:25，早于计划的 21:53，正确标签是 on_time。

现在不要根据这两条结果修改提示词或评分规则。先按已固定的设置跑完整验证集，看看问题出现的频率。

#6：保存完整验证集的原始模型预测

对 200 条验证样本逐条推理，保存模型原始回答和标准答案。

本步骤只生成并保存结果，暂不计算总分。保留原始回答很重要：例如代码围栏会影响严格 JSON 有效率，但仍可用于分析字段内容是否正确。

模型输入仅包含 system 和 user 消息；assistant 标准答案只写入结果文件用于后续评分。

In [ ]:
from pathlib import Path
from tqdm.auto import tqdm
import json
import time

# 将结果保存在 Colab 工作目录
output_path = Path("/content/phase2_qwen_base_val_predictions.jsonl")
start_time = time.time()

# 每完成一条就写入文件，便于保留逐条原始证据
with output_path.open("w", encoding="utf-8") as f:
    for sample in tqdm(val_data, desc="验证集推理"):
        # generate_answer 内部只向模型传入 system 和 user
        prediction = generate_answer(sample)

        # 标准答案用于之后评分，不参与模型生成
        result = {
            "sample_id": sample["sample_id"],
            "source_load_id": sample["source_load_id"],
            "task": sample["task"],
            "template_id": sample["template_id"],
            "prediction": prediction,
            "gold": sample["messages"][2]["content"],
        }

        # 保留模型回答的原始文本，包括可能出现的代码围栏
        f.write(json.dumps(result, ensure_ascii=False) + "\n")
        f.flush()

print("完成样本数：", len(val_data))
print("总耗时（秒）：", round(time.time() - start_time, 1))
print("结果文件：", output_path)

#8. 保存结果并评测 Task A

本步骤先下载 200 条原始预测记录，再评测其中 100 条信息提取样本。

评分坚持预先确定的严格规则：

- 回答必须能直接解析为 JSON 对象；带 Markdown 代码围栏的回答不算有效 JSON。
- 结构正确表示恰好包含指定的 8 个字段。
- 字段准确率以 100 × 8 = 800 个字段为分母；无效 JSON 的字段全部记错。
- 缺失字段准确率检查标准答案为 null 的字段是否也输出 null。

这些指标分别反映输出格式、字段结构、内容准确性和“不编造缺失信息”的能力。

In [ ]:
from google.colab import files
from pathlib import Path
import json
import math

# 先下载原始预测记录，避免 Colab 会话结束后丢失
output_path = Path("/content/phase2_qwen_base_val_predictions.jsonl")
assert output_path.exists(), "找不到预测结果文件"
files.download(str(output_path))

# 重新读取已保存的逐条预测，而不是重新运行模型
with output_path.open("r", encoding="utf-8") as f:
    results = [json.loads(line) for line in f if line.strip()]

# 确认文件完整，且只包含本次验证集的样本
assert len(results) == 200, f"结果只有 {len(results)} 条"
assert {r["sample_id"] for r in results} == {
    s["sample_id"] for s in val_data
}, "结果文件与验证集样本不一致"

a_results = [r for r in results if r["task"] == "A"]
assert len(a_results) == 100

def parse_json_object(text):
    """严格解析原始回答；代码围栏等额外文字会导致解析失败。"""
    try:
        value = json.loads(text)
    except (json.JSONDecodeError, TypeError):
        return None

    # 本任务要求输出 JSON 对象，而不是列表、数字等其他 JSON 值
    return value if isinstance(value, dict) else None

def field_matches(predicted, expected):
    """比较单个字段；数字允许极小的浮点表示误差。"""
    if isinstance(expected, (int, float)) and not isinstance(expected, bool):
        return (
            isinstance(predicted, (int, float))
            and not isinstance(predicted, bool)
            and math.isclose(
                predicted, expected, rel_tol=0, abs_tol=1e-6
            )
        )

    # 字符串要求内容一致；null 只有对应 Python None 才算正确
    return type(predicted) is type(expected) and predicted == expected

valid_json = 0
correct_schema = 0
correct_fields = 0
null_fields = 0
correct_null_fields = 0
fenced_outputs = 0

for row in a_results:
    raw_answer = row["prediction"].strip()
    gold = json.loads(row["gold"])

    # 记录代码围栏出现次数，仅用于错误分析，不放宽主评分
    if raw_answer.startswith("```") and raw_answer.endswith("```"):
        fenced_outputs += 1

    prediction = parse_json_object(raw_answer)

    if prediction is not None:
        valid_json += 1

        if set(prediction.keys()) == set(A_FIELDS):
            correct_schema += 1

    # 遍历标准答案的全部 8 个字段；解析失败时均记为错误
    for field in A_FIELDS:
        expected = gold[field]

        if expected is None:
            null_fields += 1

        if prediction is not None and field in prediction:
            if field_matches(prediction[field], expected):
                correct_fields += 1

                if expected is None:
                    correct_null_fields += 1

total_fields = len(a_results) * len(A_FIELDS)

print("Task A 样本数：", len(a_results))
print(f"JSON 有效率：{valid_json}/{len(a_results)} = {valid_json / len(a_results):.1%}")
print(f"结构正确率：{correct_schema}/{len(a_results)} = {correct_schema / len(a_results):.1%}")
print(f"字段准确率：{correct_fields}/{total_fields} = {correct_fields / total_fields:.1%}")

if null_fields:
    print(
        f"缺失字段准确率：{correct_null_fields}/{null_fields}"
        f" = {correct_null_fields / null_fields:.1%}"
    )

print("带代码围栏的回答数：", fenced_outputs)

# 8. 分析 Task A 的代码围栏问题

严格评测发现，100 条 Task A 回答全部带 Markdown 代码围栏，导致直接 JSON 解析失败。

本步骤增加一组诊断指标：只对完整代码围栏做格式剥离，再检查内部 JSON 的结构、字段值和 null。它用于区分“不会提取字段”和“字段正确但输出格式不合要求”。

正式基线仍保留上一步的严格成绩；诊断成绩单独报告。

In [ ]:
import re
import json

recovered_json = 0
recovered_schema = 0
recovered_fields = 0
recovered_null_fields = 0

for row in a_results:
    raw_answer = row["prediction"].strip()
    gold = json.loads(row["gold"])

    # 只识别“整个回答恰好被一层 Markdown 代码围栏包住”的情况
    # 不改动围栏内部的字段或数值
    match = re.fullmatch(
        r"```(?:json)?\s*\n?(.*?)\s*```",
        raw_answer,
        flags=re.DOTALL | re.IGNORECASE,
    )

    # 没有完整围栏时，仍使用原始回答
    content = match.group(1).strip() if match else raw_answer

    try:
        prediction = json.loads(content)
    except json.JSONDecodeError:
        continue

    # 本任务要求 JSON 对象
    if not isinstance(prediction, dict):
        continue

    recovered_json += 1

    # 检查是否恰好包含指定的 8 个字段
    if set(prediction.keys()) == set(A_FIELDS):
        recovered_schema += 1

    # 比较每个字段的值
    for field in A_FIELDS:
        if field not in prediction:
            continue

        if field_matches(prediction[field], gold[field]):
            recovered_fields += 1

            # 单独统计应填 null 的字段
            if gold[field] is None:
                recovered_null_fields += 1

print("Task A 诊断指标（仅剥离完整代码围栏）：")
print(f"可解析 JSON：{recovered_json}/100 = {recovered_json / 100:.1%}")
print(f"结构正确：{recovered_schema}/100 = {recovered_schema / 100:.1%}")
print(f"字段正确：{recovered_fields}/800 = {recovered_fields / 800:.1%}")
print(
    f"null 字段正确：{recovered_null_fields}/{null_fields}"
    f" = {recovered_null_fields / null_fields:.1%}"
)


**通俗说，Task A 的模型会“填表”，但没有按要求“交表”**。

它在 100 道题里都把 JSON 包进了 Markdown 代码框（```json ... ```）。我们的系统要求直接收到 JSON，所以这 100 个回答都不能直接被严格解析，正式 JSON 有效率是 0%。

但把外面的代码框拿掉后，里面的内容并不差：800 个字段答对了 757 个（94.6%）。其中需要填 null 的 40 个字段答对了 33 个，说明它有时还会猜测题目没提供的信息。

所以 **Task A 主要有两个改进点**：严格按格式输出，以及遇到缺失信息时稳定地填 null。我们后续要分别衡量这两点是否因微调而改善。

这也给后续 LoRA 微调一个清晰目标：提高严格 JSON 有效率，并改善缺失信息处理；不能只说“字段准确率从 0% 提升”，因为 0% 主要由解析规则造成。

# Task A 错误分析

原始模型的 100 条 Task A 回答全部带代码围栏。剥离围栏后，800 个字段中有 757 个正确。

本步骤统计每个字段的错误次数，并查看缺失信息相关的错误案例。这里仅分析已有预测，不重新运行模型，也不修改正式基线成绩。

完成标准：得到字段错误统计，以及具体的 null 错误案例。

In [ ]:
from collections import Counter
import json
import re

# 统计各字段的错误次数，并保存具体案例
field_errors = Counter()
error_examples = []
null_errors = []

for row in a_results:
    raw_answer = row["prediction"].strip()
    gold = json.loads(row["gold"])

    # 仅为分析内容，剥离包住整个回答的代码围栏
    match = re.fullmatch(
        r"```(?:json)?\s*\n?(.*?)\s*```",
        raw_answer,
        flags=re.DOTALL | re.IGNORECASE,
    )
    content = match.group(1).strip() if match else raw_answer
    prediction = json.loads(content)

    # 逐字段比较模型回答和标准答案
    for field in A_FIELDS:
        expected = gold[field]
        actual = prediction.get(field)

        if not field_matches(actual, expected):
            field_errors[field] += 1

            case = {
                "sample_id": row["sample_id"],
                "field": field,
                "标准答案": expected,
                "模型回答": actual,
            }
            error_examples.append(case)

            # 单独收集应填 null、但模型没有正确填写的情况
            if expected is None:
                null_errors.append(case)

print("各字段错误次数：")
for field in A_FIELDS:
    print(f"  {field}: {field_errors[field]}")

print("\n错误字段总数：", sum(field_errors.values()))
print("null 错误数：", len(null_errors))

print("\n全部 null 错误案例：")
for case in null_errors:
    print(case)

print("\n其他错误示例（最多 8 个）：")
for case in [c for c in error_examples if c["标准答案"] is not None][:8]:
    print(case)

# Task A 提示词对照实验

已有基线显示：模型常给 JSON 加代码围栏，并在运单编号、计划/实际时间和缺失字段上犯错。

本实验只加强 system 消息中的输出要求，不修改模型权重、用户题目或生成参数。先选 10 条验证样本，包含已发现的错误案例和普通案例。

重点观察：回答能否直接解析为 JSON、8 个字段是否正确，以及缺失信息是否使用不带引号的 null。原始基线成绩保持不变。

In [ ]:
import json

# 在原有系统要求后增加通用规则，不写入任何验证集答案
EXTRA_INSTRUCTIONS = (
    "\n输出格式必须严格遵守：第一个字符是 {，最后一个字符是 }。"
    "不要输出 ```、json 标记或任何 JSON 对象之外的文字。"
    "字段未在用户记录中出现时，填写 JSON null，不要写字符串 \"null\"，也不要猜测。"
    "运单编号、车辆编号和时间只从用户记录原样复制。"
    "计划交付时间与实际交付时间分别对应各自字段，不要交换。"
)

def generate_answer_v2(sample):
    # 复制原有 system 消息并追加规则；用户题目保持原样
    prompt_messages = [
        {
            "role": "system",
            "content": sample["messages"][0]["content"] + EXTRA_INSTRUCTIONS,
        },
        sample["messages"][1],
    ]

    # 使用与原始基线相同的聊天模板和生成设置
    inputs = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=True,
        add_generation_prompt=True,
        enable_thinking=False,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)

    # 本阶段只做推理，不更新模型参数
    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    # 只返回模型新生成的内容
    new_tokens = output_ids[0, inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

# 包含编号、时间和 null 的已知错误案例
target_ids = [
    "LOAD00019804_A",
    "LOAD00003026_A",
    "LOAD00028128_A",
    "LOAD00066162_A",
    "LOAD00048633_A",
    "LOAD00028326_A",
]

selected = [
    s for s in val_data
    if s["sample_id"] in target_ids
]

# 再补充 4 条普通样本，检查提示词是否产生新的问题
selected_ids = {s["sample_id"] for s in selected}
selected += [
    s for s in val_data
    if s["task"] == "A" and s["sample_id"] not in selected_ids
][:4]

for sample in selected:
    answer = generate_answer_v2(sample)
    parsed = parse_json_object(answer)
    gold = json.loads(sample["messages"][2]["content"])

    # 严格 JSON 解析失败时，字段得分记为 0
    correct_fields = sum(
        field in parsed and field_matches(parsed[field], gold[field])
        for field in A_FIELDS
    ) if parsed is not None else 0

    print(
        sample["sample_id"],
        "| 严格 JSON：", parsed is not None,
        "| 正确字段：", f"{correct_fields}/8",
    )
    print("回答：", answer[:900])
    print()

# Task A 新提示词的完整验证集对照

10 条试运行中，模型均直接输出了可解析的 JSON，但仍存在字段错误。

本步骤固定当前新提示词，对全部 100 条 Task A 验证样本推理。保存逐条回答，再按原先相同的严格规则计算 JSON 有效率、结构正确率、字段准确率和 null 准确率。

原始提示词的结果不会覆盖。两版结果分开保存，用于比较“只修改提示词”带来的变化。

In [ ]:
from pathlib import Path
from tqdm.auto import tqdm
from google.colab import files
import json
import time

# 只选择 Task A；顺序沿用原验证集
a_samples = [s for s in val_data if s["task"] == "A"]
assert len(a_samples) == 100

# 新提示词的结果使用新文件名，不覆盖原始基线
v2_path = Path("/content/phase2_qwen_base_val_A_prompt_v2.jsonl")
start_time = time.time()

with v2_path.open("w", encoding="utf-8") as f:
    for sample in tqdm(a_samples, desc="Task A 新提示词推理"):
        # 使用刚才试验过的 v2 提示词函数
        prediction = generate_answer_v2(sample)

        # 保存原始回答和标准答案，方便逐条复查
        record = {
            "sample_id": sample["sample_id"],
            "task": "A",
            "prompt_version": "v2",
            "prediction": prediction,
            "gold": sample["messages"][2]["content"],
        }
        f.write(json.dumps(record, ensure_ascii=False) + "\n")
        f.flush()

# 从保存的文件重新读取结果，再计算指标
with v2_path.open("r", encoding="utf-8") as f:
    v2_results = [json.loads(line) for line in f if line.strip()]

assert len(v2_results) == 100
assert {r["sample_id"] for r in v2_results} == {
    s["sample_id"] for s in a_samples
}

valid_json = 0
correct_schema = 0
correct_fields = 0
null_fields_v2 = 0
correct_null_fields = 0

for row in v2_results:
    gold = json.loads(row["gold"])

    # 沿用原先的严格解析规则；不会自动去掉代码围栏
    prediction = parse_json_object(row["prediction"].strip())

    if prediction is not None:
        valid_json += 1
        if set(prediction.keys()) == set(A_FIELDS):
            correct_schema += 1

    for field in A_FIELDS:
        expected = gold[field]

        if expected is None:
            null_fields_v2 += 1

        # 无效 JSON 或缺失字段直接记错
        if (
            prediction is not None
            and field in prediction
            and field_matches(prediction[field], expected)
        ):
            correct_fields += 1
            if expected is None:
                correct_null_fields += 1

print("Task A 新提示词全量结果：")
print(f"严格 JSON 有效率：{valid_json}/100 = {valid_json / 100:.1%}")
print(f"结构正确率：{correct_schema}/100 = {correct_schema / 100:.1%}")
print(f"字段准确率：{correct_fields}/800 = {correct_fields / 800:.1%}")
print(
    f"null 准确率：{correct_null_fields}/{null_fields_v2}"
    f" = {correct_null_fields / null_fields_v2:.1%}"
)
print("总耗时（秒）：", round(time.time() - start_time, 1))
print("结果文件：", v2_path)

# 下载新结果；原始基线文件仍单独保留
files.download(str(v2_path))

**Task A 的格式问题已经解决**：新提示词在全部 100 条验证样本上都直接输出了合法 JSON，结构也全部正确。

新提示词除了修正围栏，还让内容多答对了 27 个字段。但 null 没有改善。建议固定这个 v2 提示词，不继续针对同一验证集反复修改；把缺失信息处理作为后续微调要解决的具体问题。之后比较 LoRA 效果时，原始模型和 LoRA 模型都使用同一个 v2 提示词。

#Task B 原始模型评分

读取已保存的 100 条 Task B 预测，计算严格格式有效率、Accuracy，以及 delayed 和 on_time 各自的 Precision、Recall、F1。

验证集有 67 条 delayed、33 条 on_time。只看 Accuracy 容易掩盖模型偏向多数类的问题，因此同时报告 Macro-F1 和每类的 Recall。

格式无效或标签不属于允许的两类时，该样本记为预测错误。

In [ ]:
import json
from collections import Counter

# 使用原始基线已保存的结果，不重新运行模型
b_results = [r for r in results if r["task"] == "B"]
assert len(b_results) == 100

gold_labels = []
pred_labels = []

for row in b_results:
    # 读取标准答案
    gold = json.loads(row["gold"])["delay_label"]

    # 严格解析原始回答；不剥离代码围栏
    parsed = parse_json_object(row["prediction"].strip())

    # 只有结构和标签都正确，才接受模型预测
    if (
        parsed is not None
        and set(parsed.keys()) == {"delay_label"}
        and parsed["delay_label"] in B_LABELS
    ):
        pred = parsed["delay_label"]
    else:
        pred = None

    gold_labels.append(gold)
    pred_labels.append(pred)

# 无效输出也计入 100 条总数，并视为预测错误
correct = sum(g == p for g, p in zip(gold_labels, pred_labels))
valid = sum(p is not None for p in pred_labels)

print(f"严格格式有效率：{valid}/100 = {valid / 100:.1%}")
print(f"Accuracy：{correct}/100 = {correct / 100:.1%}")

f1_values = []

for label in B_LABELS:
    # 将当前类别分别视为正类
    tp = sum(g == label and p == label for g, p in zip(gold_labels, pred_labels))
    fp = sum(g != label and p == label for g, p in zip(gold_labels, pred_labels))
    fn = sum(g == label and p != label for g, p in zip(gold_labels, pred_labels))

    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0

    f1_values.append(f1)
    print(
        f"{label}: Precision={precision:.3f}, "
        f"Recall={recall:.3f}, F1={f1:.3f}"
    )

# 两类 F1 等权平均
print(f"Macro-F1：{sum(f1_values) / 2:.3f}")
print("预测分布：", dict(Counter(pred_labels)))

task B 的结果很明确：模型把 100 条全部判成 delayed。所以 67% Accuracy 恰好等于验证集中 delayed 的占比；它一条 on_time 都没识别出来，Macro-F1 只有 0.401。目前不能说它学会了延误判断。

还要客观看待任务本身：当前 Task B 只需比较“实际时间是否晚于计划时间”。如果两项时间都在输入里，普通程序也能按规则判断。因此后续我们应加入一个日期比较的规则基线，避免把模型学会简单规则描述成复杂的风险推理成果。

先做一次有限的 Task B 提示词对照：用不来自数据集的两个小例子说明时间比较，各选 5 条 on_time 和 delayed 试运行。原始 67% 成绩保留。

# Task B 提示词对照试验

原始 Qwen 将 100 条记录全部预测为 delayed，未识别任何 on_time 样本。

本步骤只修改 Task B 的系统提示词，加入明确的时间比较步骤和两个自编示例。模型权重、用户题目和生成参数保持不变。

先分别试运行 5 条 on_time 和 5 条 delayed，检查模型是否能识别两类。此处是诊断试验，不作为完整验证集成绩。

In [ ]:
import json
import torch

# 两个示例使用自编时间，不取自训练集、验证集或测试集
B_EXTRA_INSTRUCTIONS = (
    "\n判断时先比较完整的日期和时间，不要根据某个类别出现的频率猜测。"
    "实际时间严格晚于计划时间才是 delayed；早于或等于计划时间是 on_time。"
    "例如：计划 2020-01-01 10:00，实际 2020-01-01 11:00，标签是 delayed。"
    "计划 2020-01-01 11:00，实际 2020-01-01 10:00，标签是 on_time。"
    "最终只输出一个 JSON 对象，格式为 {\"delay_label\":\"delayed\"} "
    "或 {\"delay_label\":\"on_time\"}，不要添加其他文字。"
)

def generate_b_v2(sample):
    # 只加强系统要求；用户题目保持原样，标准答案不传给模型
    prompt_messages = [
        {
            "role": "system",
            "content": sample["messages"][0]["content"] + B_EXTRA_INSTRUCTIONS,
        },
        sample["messages"][1],
    ]

    # 保持与原始基线相同的聊天模板、非思考模式和生成参数
    inputs = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=True,
        add_generation_prompt=True,
        enable_thinking=False,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    # 只解码新生成的回答
    new_tokens = output_ids[0, inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

# 各取 5 条，防止“全部预测 delayed”在小样本里看似有效
on_time_samples = [
    s for s in val_data
    if s["task"] == "B"
    and json.loads(s["messages"][2]["content"])["delay_label"] == "on_time"
][:5]

delayed_samples = [
    s for s in val_data
    if s["task"] == "B"
    and json.loads(s["messages"][2]["content"])["delay_label"] == "delayed"
][:5]

selected_b = on_time_samples + delayed_samples

for sample in selected_b:
    answer = generate_b_v2(sample)
    gold = json.loads(sample["messages"][2]["content"])["delay_label"]
    parsed = parse_json_object(answer)
    pred = parsed.get("delay_label") if parsed else None

    print(
        sample["sample_id"],
        "| 标准：", gold,
        "| 预测：", pred,
        "| 正确：", pred == gold,
    )
    print("原始回答：", answer[:300])

# 检查 Task B 的时间表达

提示词对照中，模型仍未识别 5 条 on_time 样本。

本步骤查看这些题目的原文、计划时间、实际时间和标准标签，确认题目是否提供了足够信息，以及时间的表达方式。

这里只查看已有数据，不重新运行模型。

In [ ]:
import json

# 查看刚才试运行的 5 条 on_time 题目
for sample in on_time_samples:
    # user 消息是模型实际看到的题目
    question = sample["messages"][1]["content"]

    # assistant 消息是标准答案，只供我们检查
    gold = json.loads(sample["messages"][2]["content"])

    print(f"\n===== {sample['sample_id']} =====")
    print("题目：", question)
    print("标准标签：", gold["delay_label"])

#Task B 日期比较规则基线

Task B 的当前目标是判断实际交付是否晚于计划时间。

本步骤从题目中识别“计划”和“实际”各自对应的时间，再用 Python 比较。先检查 100 条题目能否全部正确提取时间；若有无法识别的题目，列出案例，不跳过后计算成绩。

规则基线用于衡量任务本身的难度，并与原始 Qwen 的结果对照。

In [ ]:
import re
from datetime import datetime

# 匹配题目中完整的日期时间，包括可选的小数秒
TIME_PATTERN = re.compile(
    r"\d{4}-\d{2}-\d{2} "
    r"\d{2}:\d{2}:\d{2}(?:\.\d+)?"
)

def predict_delay_by_rule(question):
    times = {}

    for match in TIME_PATTERN.finditer(question):
        # 查看当前时间左边的文字，寻找最近的“计划”或“实际”
        prefix = question[max(0, match.start() - 60):match.start()]
        markers = list(re.finditer(r"计划|实际", prefix))

        if not markers:
            return None

        # 最近的标记决定这个时间属于计划还是实际
        role = markers[-1].group()
        if role in times:
            return None

        times[role] = datetime.fromisoformat(match.group())

    # 必须恰好识别出计划和实际两个时间，才能给出判断
    if set(times) != {"计划", "实际"}:
        return None

    return (
        "delayed"
        if times["实际"] > times["计划"]
        else "on_time"
    )

rule_results = []
unresolved = []

for sample in val_data:
    if sample["task"] != "B":
        continue

    # 规则程序只读取用户题目，不读取标准答案来作预测
    question = sample["messages"][1]["content"]
    predicted = predict_delay_by_rule(question)

    # 标准答案只用于完成预测后的核对
    gold = json.loads(sample["messages"][2]["content"])["delay_label"]

    rule_results.append((sample["sample_id"], predicted, gold))

    if predicted is None:
        unresolved.append(sample["sample_id"])

print("Task B 总数：", len(rule_results))
print("成功识别计划和实际时间：", len(rule_results) - len(unresolved))
print("无法识别的样本数：", len(unresolved))
print("无法识别的样本编号（最多 10 个）：", unresolved[:10])

# 只有全部题目都成功识别，才报告完整规则基线成绩
if not unresolved:
    correct = sum(pred == gold for _, pred, gold in rule_results)
    print(f"规则基线 Accuracy：{correct}/100 = {correct / 100:.1%}")

    wrong = [
        (sample_id, pred, gold)
        for sample_id, pred, gold in rule_results
        if pred != gold
    ]
    print("规则判断错误数：", len(wrong))
    print("错误案例（最多 10 个）：", wrong[:10])

# 完成 Task B 延误判断模块

Task B 的标签由一条明确规则决定：实际交付时间晚于计划时间为 delayed，否则为 on_time。

本模块从题目中识别两项时间，使用 Python 比较，并输出标准 JSON。原始 Qwen 的预测结果单独保留，不与规则结果混用。

完成标准：100 条验证样本都能得到合法标签，逐条结果保存为 JSONL，并报告规则模块的验证集准确率。测试集暂不使用。

In [ ]:
from pathlib import Path
from google.colab import files
import json

# 单独保存规则模块的结果，不覆盖 Qwen 的原始预测
rule_output_path = Path("/content/phase2_taskB_rule_val_predictions.jsonl")

rule_correct = 0
rule_total = 0

with rule_output_path.open("w", encoding="utf-8") as f:
    for sample in val_data:
        if sample["task"] != "B":
            continue

        # 规则只读取用户提供的交付记录
        question = sample["messages"][1]["content"]
        label = predict_delay_by_rule(question)

        # 无法识别时间时明确报错，避免悄悄猜一个标签
        if label is None:
            raise ValueError(f"无法判断：{sample['sample_id']}")

        # 按 Task B 要求生成标准 JSON 对象
        prediction = {"delay_label": label}

        # 标准答案只在预测完成后用于核对
        gold = json.loads(sample["messages"][2]["content"])
        rule_correct += prediction == gold
        rule_total += 1

        # 保存每条结果，供后续报告复查
        record = {
            "sample_id": sample["sample_id"],
            "method": "datetime_rule",
            "prediction": prediction,
            "gold": gold,
        }
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

assert rule_total == 100
print(f"Task B 规则模块验证集 Accuracy：{rule_correct}/{rule_total} "
      f"= {rule_correct / rule_total:.1%}")
print("结果文件：", rule_output_path)

# 下载逐条记录，避免 Colab 会话结束后丢失
files.download(str(rule_output_path))

Task B 的规则模块已完成验证：100/100，Accuracy 100%。 逐条结果也已保存。这个数字只代表验证集上的日期比较程序；原始 Qwen 的 Task B 成绩仍是 67% Accuracy、0.401 Macro-F1，两者不能合并。

## FleetMind-LLM｜Phase 2 阶段结论

**Phase 2 完成了原始 Qwen3-0.6B 的验证集基线评测，并为 Task B 建立了独立的日期比较规则模块。** 本阶段使用 200 条验证指令：Task A、B 各 100 条，来自 100 条运单。尚未进行 SFT 或 LoRA 训练，测试集也尚未使用。

| 任务与方法 | 验证集结果 | 主要发现 |
|---|---:|---|
| Task A：原始提示词，严格评分 | JSON 有效率 **0/100**；字段 **0/800** | 100 条回答均带代码围栏，不能直接解析为 JSON |
| Task A：原始回答剥离围栏后的诊断 | 字段 **757/800（94.6%）**；`null` **33/40（82.5%）** | 模型有提取能力；严格评分的 0% 主要来自输出格式 |
| Task A：改进提示词，全量重新评测 | JSON **100/100**；结构 **100/100**；字段 **784/800（98.0%）**；`null` **33/40（82.5%）** | 格式问题解决，字段多答对 27 个；缺失信息处理仍未改善 |
| Task B：原始 Qwen | Accuracy **67%**；Macro-F1 **0.401**；`on_time` Recall **0** | 100 条全部预测为 `delayed`，未学会区分两类 |
| Task B：日期比较规则 | Accuracy **100/100** | 在当前明确的时间比较任务上，规则程序能完成判断 |


**核心判断**：Task A 是后续 LoRA 最值得重点检验的部分，尤其要看结构化输出和缺失字段的 `null` 是否更稳定。Task B 当前只是明确的交付时间比较任务；规则模块解决了业务判断，但它的 100% **不是模型成绩**。LoRA 仍可用 Task B 检验模型能否摆脱“全部预测 delayed”，报告时必须与规则成绩分开。

**证据与限制**：两版 Qwen 的逐条预测、Task B 规则模块的逐条结果都已保存。Task A 新提示词是在验证集上改进的，所有上述分数也都是**验证集结果**，不能当作最终泛化成绩。后续固定提示词和评分代码，再用保留的测试集做最终比较。